# Exercici de regressió polinòmica regularitzada

```python

y =  2 * df["Var_1"] +  df["Var_2"]**2 + (df["Var_3"] - 1)**3 + 3 * np.sin(5*df["Var_4"]) + df["Var_50"] * df["Var_60"]  + np.where(df['Var_96'] > 0.5, 10 * df['Var_96'], 0)                

```

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
df = pd.read_csv("dataset_sintetic_reg.csv")
df.describe()

,Var_1,Var_2,Var_3,Var_4,Var_5,Var_6,Var_7,Var_8,Var_9,Var_10,...,Var_92,Var_93,Var_94,Var_95,Var_96,Var_97,Var_98,Var_99,Var_100,y
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,...,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.034691,-0.008023,-0.041958,0.048109,0.044938,-0.029735,-0.042847,0.026396,-0.010257,-0.008596,...,0.012133,0.035319,-0.000800,0.173456,-0.017331,0.052052,-0.033863,0.012640,-0.027940,-1.305781
std,1.696384,1.710395,1.833256,1.717389,1.698877,1.705547,1.847132,1.786225,1.712326,1.717888,...,1.848157,1.741720,1.733986,8.481920,1.755736,1.816463,1.679320,1.832581,1.733034,34.382294
min,-5.682832,-5.517704,-5.398888,-5.455223,-4.988883,-5.480520,-6.113626,-5.547733,-5.105486,-4.713854,...,-6.321259,-5.018670,-4.900278,-28.414158,-6.088760,-5.213307,-4.328812,-6.017352,-5.622204,-244.894154
25%,-1.132145,-1.230294,-1.220841,-1.027569,-1.056184,-1.118479,-1.266795,-1.109322,-1.200873,-1.254341,...,-1.217312,-1.076770,-1.154857,-5.660723,-1.142215,-1.272359,-1.191569,-1.166035,-1.115622,-6.533973
50%,0.046094,-0.009192,-0.086197,0.000330,0.021624,0.026099,-0.089576,0.002909,-0.001816,0.030930,...,0.055818,0.060162,-0.024461,0.230469,0.037250,0.039566,-0.080569,0.047744,0.004790,2.775009
75%,1.232034,1.208792,1.096686,1.251876,1.170946,1.121502,1.248872,1.220824,1.094037,1.210216,...,1.256182,1.260854,1.239202,6.160168,1.210406,1.307182,1.084376,1.214401,1.101616,13.211569
max,6.097426,6.114505,7.263857,4.824848,6.420349,5.041985,5.783461,5.256749,6.183809,4.893511,...,6.217616,5.064019,5.107291,30.487130,5.726962,5.697885,5.792467,7.195059,5.039393,250.064473


In [6]:
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

X = df_final.drop("y", axis=1)
y = df_final["y"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(3, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)
X_poly_feature_name = poly.get_feature_names_out()
df_poly = pd.DataFrame(X_train_poly, columns=X_poly_feature_name)
print(df_poly.columns)
df_poly.head()


Index(['Var_1', 'Var_2', 'Var_3', 'Var_4', 'Var_5', 'Var_6', 'Var_7', 'Var_8',
       'Var_9', 'Var_10',
       ...
       'Var_98^3', 'Var_98^2 Var_99', 'Var_98^2 Var_100', 'Var_98 Var_99^2',
       'Var_98 Var_99 Var_100', 'Var_98 Var_100^2', 'Var_99^3',
       'Var_99^2 Var_100', 'Var_99 Var_100^2', 'Var_100^3'],
      dtype='str', length=176850)


,Var_1,Var_2,Var_3,Var_4,Var_5,Var_6,Var_7,Var_8,Var_9,Var_10,...,Var_98^3,Var_98^2 Var_99,Var_98^2 Var_100,Var_98 Var_99^2,Var_98 Var_99 Var_100,Var_98 Var_100^2,Var_99^3,Var_99^2 Var_100,Var_99 Var_100^2,Var_100^3
0,-0.669374,1.432518,-1.005721,0.165765,-1.754613,1.570868,-1.384675,1.143971,-1.212538,2.626562,...,-3.973888,1.462977,-0.526342,-0.538591,0.193771,-0.069714,0.198281,-0.071336,0.025665,-0.009234
1,-4.558313,-0.933279,1.621977,-2.421173,-0.191625,-2.179202,0.626004,3.347371,3.163880,-1.150329,...,-5.980684,-0.236014,-2.550631,-0.009314,-0.100655,-1.087788,-0.000368,-0.003972,-0.042927,-0.463918
2,2.204588,0.827569,3.276971,0.411086,1.720068,1.315117,0.814472,-2.339990,1.847949,-0.699958,...,-0.304884,-0.460228,0.974987,-0.694722,1.471761,-3.117907,-1.048696,2.221650,-4.706539,9.970745
3,-1.315511,-1.533737,1.106120,-2.529998,-0.858091,0.986197,0.372320,0.170212,1.081158,2.389669,...,-0.909025,-0.974016,2.162049,-1.043653,2.316625,-5.142276,-1.118270,2.482254,-5.509925,12.230529
4,-2.267846,-5.517704,0.595040,3.866245,-0.446880,0.230393,1.445032,2.431717,0.046571,0.500459,...,14.888172,-8.497469,4.585776,4.849956,-2.617346,1.412487,-2.768127,1.493858,-0.806181,0.435067


PolynomialFeatures ha generat moltes columnes que no van a servir per a res. Si entrenem un model Lineal normal pot ser que quede inestable. 

> Entrena Ridge, Lasso i ElasticNet i treu les mètriques. 

> Amb el model ElasticNet imprimeix les columnes que tenen un coeficient absolut major que 0.1 (deuria funcionar perquè són dades estandaritzades) i la quantitat de columnes descartades. Interpreta el resultat.

> Fes un anàlisi ANOVA amb SelectKBest sobre `df_poly` i observa si detecta les mateixes variables que ElasticNet 
